# Protein contacts from coevolution (DCA)

*bioinformatics · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/bioinformatics/coevolution_contacts)

Residues that touch in a folded protein mutate together over evolution. We fold a toy protein on a cubic lattice, plant couplings $J_{ij}(a,b)$ on its contacts, sample a multiple sequence alignment from the Potts model by Gibbs sampling, then recover the contacts from the alignment alone: pair statistics by one-hot einsum, direct couplings by mean-field DCA $J \approx -C^{-1}$ (or pseudo-likelihood with autograd), and an APC-corrected Frobenius score, checked against the true contact map.

**Vector form**

$$P(\mathbf s) = \frac{1}{Z}\exp\!\Big(\sum_i h_i(s_i) + \sum_{i<j} J_{ij}(s_i, s_j)\Big),\qquad \mathbf C = \langle \mathbf x\,\mathbf x^{\mathsf T}\rangle - \langle\mathbf x\rangle\langle\mathbf x\rangle^{\mathsf T},\qquad \mathbf J^{\mathrm{MF}} = -\mathbf C^{-1}$$

**Index (tensor) form**

$$E_{nia} = h_{ia} + J_{iajb}\,x_{njb} \qquad f_{ia} = \tfrac1N x_{nia} \qquad C_{iajb} = \tfrac1N x_{nia}x_{njb} - f_{ia}f_{jb} \qquad F_{ij} = \sqrt{J_{iajb}J_{iajb}} \qquad F^{\mathrm{APC}}_{ij} = F_{ij} - \frac{\bar F_{i\cdot}\,\bar F_{\cdot j}}{\bar F_{\cdot\cdot}}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $r_{ia},\ \mathcal C_{ij}$ | `lattice_fold(dims), contact_map(coords, 3)` | (L, 3), (L, L) | a serpentine chain filling an $X\times Y\times Z$ box; a contact is a lattice neighbour at least 3 apart along the chain |
| $h_{ia},\ J_{iajb}$ | `planted_potts(contacts, q, coupling, 1.0, seed)` | (L, q), (L, q, L, q) | the "true" model: random fields, and random $q\times q$ coupling blocks only where $\mathcal C_{ij} = 1$ |
| $E_{nia}$ | `h + einsum('iajb,njb->nia', J, x)` | (N, L, q) | local field: $j, b$ summed over every other column and letter of sequence $n$ |
| $s_{ni} \sim \operatorname{softmax}_a E_{nia}$ | `gibbs_sample(h, J, colour, N, sweeps, seed)` | (N, L) | checkerboard Gibbs: contacts join opposite colours, so half of the sites are resampled at once, in all $N$ chains |
| $x_{nia}$ | `one_hot(seqs, q)` | (N, L, q) | the alignment as a one-hot tensor |
| $f_{ia},\ f_{iajb}$ | `einsum('nia->ia', x) / N, einsum('nia,njb->iajb', x, x) / N` | (L, q), (L, q, L, q) | $n$ summed: single and pair frequencies; summing $f_{iajb}$ over $b$ gives back $f_{ia}$ |
| $C_{iajb}$ | `f2 - einsum('ia,jb->iajb', f1, f1)` | (L, q, L, q) | connected correlation, after mixing in a pseudocount $\lambda$ of uniform frequencies |
| $J^{\mathrm{MF}}_{iajb}$ | `-torch.linalg.inv(C[:, :q-1, :, :q-1].reshape(...))` | (L, q, L, q) | mean-field DCA: invert the $L(q-1)$-square covariance; the last letter is the reference state |
| $\log P(s_{ni}\mid s_{n,\setminus i})$ | `einsum('nia,nia->', x, log_softmax(E, -1)) / N` | () | pseudo-likelihood alternative: maximised over $h, J$ by Adam with an $L_2$ penalty |
| $F_{ij}$ | `einsum('iajb,iajb->ij', Jz, Jz).sqrt()` | (L, L) | coupling strength of a pair, $a, b$ summed, in the zero-sum gauge $J^z$ |
| $F^{\mathrm{APC}}_{ij}$ | `F - outer(F.mean(1), F.mean(0)) / F.mean()` | (L, L) | average-product correction: removes the per-column background, so rows and columns have mean 0 |

Why does this work? If residue $i$ changes its size or charge, a residue $j$ it touches must often change too, or the protein misfolds. Over many related sequences this leaves a correlation between columns $i$ and $j$ of the alignment. But correlations are transitive: if $i$ touches $j$ and $j$ touches $k$, then $i$ and $k$ also correlate without touching. Direct coupling analysis (DCA) fits a global model, the Potts model (a $q$-state Ising model), whose couplings $J_{ij}$ explain all correlations at once, so indirect ones are explained away. In the mean-field approximation the couplings are just minus the inverse of the covariance matrix (like partial correlations); pseudo-likelihood maximisation instead fits $P(s_i\mid\text{rest})$ for every column and is usually more accurate on real families. The animation grows the alignment from 20 sequences to all of them: DCA (upper triangle) improves with depth, while plain covariance (lower triangle) stalls because its false positives are indirect pairs. Here the data really come from a Potts model with known contacts, so the precision is an honest score. Real alignments are harder: a handful of letters stand in for 20 amino acids plus a gap, sequences are related by a phylogeny (real DCA down-weights near-duplicates), and only some contacts leave a signal. Coevolution is the root of today's structure prediction: plmDCA-type contacts fed early deep-learning methods, and AlphaFold 2 takes an MSA as its main input and learns its own, far richer, use of it with attention. This toy shows the statistical idea behind that input, not how AlphaFold works. Try a weaker coupling, fewer sequences, or more letters $q$.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/bioinformatics/coevolution_contacts/main.py
"""Main loop: lattice fold → planted Potts model → Gibbs-sampled MSA → DCA (or pseudo-likelihood) → contacts → plot."""

import time
from pathlib import Path

import torch

from problems._common.runtime import cli, device, fast_mode

from . import plot, solve, tensors
from .equation import precision, top_pairs

FOLDS = {"3x3x4": (3, 3, 4), "4x4x4": (4, 4, 4), "4x4x5": (4, 4, 5)}
MIN_SEP = 3  # pairs closer than this along the chain are trivially close and not scored
SEED = 0


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    fast = fast_mode()
    t0 = time.perf_counter()

    # --- the "protein": a compact lattice fold and its contacts ---
    coords = tensors.lattice_fold(FOLDS[params["fold"]], dev)  # (L, 3)
    contacts = tensors.contact_map(coords, MIN_SEP)  # (L, L) bool
    L, q = coords.shape[0], params["q"]
    n_contacts = int(contacts.sum()) // 2
    n_pairs = (L - MIN_SEP) * (L - MIN_SEP + 1) // 2
    print(f"lattice fold {params['fold']}: L = {L} residues, {n_contacts} contacts (|i−j| ≥ {MIN_SEP}), q = {q}")

    # --- evolution in a nutshell: sequences drawn from a Potts model with couplings on the contacts ---
    h, J = tensors.planted_potts(contacts, q, params["coupling"], 1.0, SEED)
    n_seq = 200 if fast else params["n_seq"]
    sweeps = 20 if fast else params["sweeps"]
    seqs = solve.gibbs_sample(h, J, tensors.parity(coords), n_seq, sweeps, SEED + 1)  # (N, L)
    x = tensors.one_hot(seqs, q)  # (N, L, q)
    print(f"MSA: {n_seq} sequences from {sweeps} checkerboard Gibbs sweeps ({time.perf_counter() - t0:.1f} s)")

    lam = params["pseudocount"]
    cov_full = solve.covariance_scores(x, lam)
    if params["method"] == "mfdca":
        # Frames: the same analysis on the first N sequences, N growing → how much data DCA needs.
        depths = torch.unique(torch.logspace(1.3, torch.log10(torch.tensor(n_seq)).item(), 6 if fast else 30).int())
        depths = depths.tolist()
        main_scores = torch.stack([solve.mfdca_scores(x[:n], lam).cpu() for n in depths])  # (S, L, L)
        cov_scores = torch.stack([solve.covariance_scores(x[:n], lam).cpu() for n in depths])
        xs, label = depths, "mean-field DCA"
    else:
        n_iter = 30 if fast else params["plm_iters"]
        _, xs, main_scores, losses = solve.train_plm(x, n_iter, 0.05, 1e-4)
        cov_scores = cov_full.cpu().expand(len(xs), L, L)
        label = "pseudo-likelihood"
        print(f"pseudo-likelihood: {n_iter} Adam steps, final loss {losses[-1]:.3f}")

    contacts_cpu = contacts.cpu()
    prec_main = [precision(top_pairs(s, L, MIN_SEP), contacts_cpu) for s in main_scores]
    prec_cov = [precision(top_pairs(s, L, MIN_SEP), contacts_cpu) for s in cov_scores]
    random_level = n_contacts / n_pairs
    for k in (L // 2, L):
        pk = precision(top_pairs(main_scores[-1], k, MIN_SEP), contacts_cpu)
        ck = precision(top_pairs(cov_scores[-1], k, MIN_SEP), contacts_cpu)
        print(f"top-{k:<3d} precision: {label} {pk:5.1%}   covariance {ck:5.1%}   random {random_level:5.1%}")
    print(f"total time {time.perf_counter() - t0:.1f} s")

    return plot.animate(
        seqs.cpu(), q, contacts_cpu, main_scores, cov_scores, xs, prec_main, prec_cov, random_level,
        label, params["method"] == "mfdca", MIN_SEP, out_dir / "coevolution_contacts.gif",
    )  # fmt: skip


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/bioinformatics/coevolution_contacts/equation.py
r"""Coevolution → tensor translation.

A multiple sequence alignment (MSA) of N sequences, L columns, q letters is a one-hot tensor x_{nia}
(sequence n has letter a at column i). The Potts model gives each sequence the probability

    P(s) ∝ exp( Σ_i h_i(s_i) + Σ_{i<j} J_ij(s_i, s_j) ).

Statistics of the alignment (n summed = average over sequences):

    f_{ia}   = ⟨x_{nia}⟩_n                     single-site frequencies
    f_{iajb} = ⟨x_{nia} x_{njb}⟩_n             pair frequencies
    C_{iajb} = f_{iajb} − f_{ia} f_{jb}        connected correlation (covariance)

Direct couplings, mean-field DCA:  J ≈ −C⁻¹ (C as an L(q−1) × L(q−1) matrix, last letter dropped as gauge).
Pseudo-likelihood:  log P(s_i | s_rest) with the local field  E_{nia} = h_{ia} + J_{iajb} x_{njb}  (j, b summed).
Contact score: Frobenius norm F_ij = ‖J_ij‖ in the zero-sum gauge, then the average-product correction (APC).
"""

import torch


def local_field(x: torch.Tensor, h: torch.Tensor, J: torch.Tensor) -> torch.Tensor:
    """E_{nia} = h_{ia} + J_{iajb} x_{njb}:  the log-weight of letter a at column i given the rest.  j, b summed.

    x: (N, L, q), h: (L, q), J: (L, q, L, q) with zero diagonal blocks → (N, L, q).
    """
    return h + torch.einsum("iajb,njb->nia", J, x)


def single_frequencies(x: torch.Tensor) -> torch.Tensor:
    """f_{ia} = (1/N) Σ_n x_{nia}.   (N, L, q) → (L, q);  n summed."""
    return torch.einsum("nia->ia", x) / x.shape[0]


def pair_frequencies(x: torch.Tensor) -> torch.Tensor:
    """f_{iajb} = (1/N) Σ_n x_{nia} x_{njb}.   (N, L, q) → (L, q, L, q);  n summed."""
    return torch.einsum("nia,njb->iajb", x, x) / x.shape[0]


def with_pseudocount(f1: torch.Tensor, f2: torch.Tensor, lam: float):
    """Mix the frequencies with a uniform distribution (weight λ) so that C is invertible.

    f_ia → (1−λ) f_ia + λ/q,   f_iajb → (1−λ) f_iajb + λ/q²  (i ≠ j),   f_iaib → f_ia δ_ab  (stay consistent).
    """
    L, q = f1.shape
    f1 = (1 - lam) * f1 + lam / q
    f2 = (1 - lam) * f2 + lam / q**2
    diag = torch.einsum("ia,ab->iab", f1, torch.eye(q, device=f1.device))  # (L, q, q): f_ia δ_ab
    idx = torch.arange(L, device=f1.device)
    f2[idx, :, idx, :] = diag
    return f1, f2


def covariance(f1: torch.Tensor, f2: torch.Tensor) -> torch.Tensor:
    """C_{iajb} = f_{iajb} − f_{ia} f_{jb}.   (L, q), (L, q, L, q) → (L, q, L, q);  outer product, nothing summed."""
    return f2 - torch.einsum("ia,jb->iajb", f1, f1)


def mean_field_couplings(C: torch.Tensor) -> torch.Tensor:
    """Mean-field DCA:  J_{iajb} = −(C⁻¹)_{iajb} for a, b < q−1  (the last letter is the reference, J = 0).

    The connected correlation matrix C (L(q−1) × L(q−1)) mixes direct and indirect effects; its inverse
    keeps only the direct ones (the same idea as partial correlations in a Gaussian graphical model).
    """
    L, q = C.shape[:2]
    Cr = C[:, : q - 1, :, : q - 1].reshape(L * (q - 1), L * (q - 1))
    Jr = -torch.linalg.inv(Cr).reshape(L, q - 1, L, q - 1)
    J = torch.zeros_like(C)
    J[:, : q - 1, :, : q - 1] = Jr
    idx = torch.arange(L, device=C.device)
    J[idx, :, idx, :] = 0.0  # no self-coupling
    return J


def zero_sum_gauge(J: torch.Tensor) -> torch.Tensor:
    """J_ij(a,b) − mean_a − mean_b + mean_ab: the gauge in which the block holds no single-site part."""
    return J - J.mean(1, keepdim=True) - J.mean(3, keepdim=True) + J.mean((1, 3), keepdim=True)


def frobenius_scores(J: torch.Tensor) -> torch.Tensor:
    """F_ij = sqrt( Σ_ab J_ij(a,b)² ) in the zero-sum gauge, diagonal 0.   (L, q, L, q) → (L, L);  a, b summed."""
    Jz = zero_sum_gauge(J)
    F = torch.einsum("iajb,iajb->ij", Jz, Jz).sqrt()
    return F.fill_diagonal_(0.0)


def apc(F: torch.Tensor) -> torch.Tensor:
    """Average-product correction  F^APC_ij = F_ij − F_i· F_·j / F_··  (means over the whole matrix).

    Columns that are very variable (or poorly sampled) have large scores with everybody; APC subtracts that
    background, and afterwards every row and column of F^APC has mean zero.
    """
    row = F.mean(1)  # F_i·
    return F - torch.outer(row, F.mean(0)) / F.mean()


def pseudo_log_likelihood(x: torch.Tensor, h: torch.Tensor, J: torch.Tensor) -> torch.Tensor:
    """(1/N) Σ_n Σ_i log P(s_ni | s_n,rest) = (1/N) Σ_{n,i,a} x_{nia} log softmax_a E_{nia}.   scalar."""
    logp = torch.log_softmax(local_field(x, h, J), dim=-1)  # (N, L, q)
    return torch.einsum("nia,nia->", x, logp) / x.shape[0]


def top_pairs(score: torch.Tensor, n_top: int, min_sep: int = 3) -> torch.Tensor:
    """The n_top highest-scoring pairs (i < j, |i−j| ≥ min_sep) as an (n_top, 2) index tensor."""
    L = score.shape[0]
    i, j = torch.triu_indices(L, L, min_sep, device=score.device)  # all pairs with j − i ≥ min_sep
    order = torch.argsort(score[i, j], descending=True)[:n_top]
    return torch.stack([i[order], j[order]], dim=1)


def precision(pairs: torch.Tensor, contacts: torch.Tensor) -> float:
    """Fraction of the predicted pairs that are true contacts."""
    return contacts[pairs[:, 0], pairs[:, 1]].float().mean().item()


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/bioinformatics/coevolution_contacts/tensors.py
"""Build the concrete tensors: a lattice fold, its contact map, the planted Potts model and one-hot sequences."""

import torch


def lattice_fold(dims: tuple[int, int, int], device: torch.device) -> torch.Tensor:
    """A compact chain on a cubic lattice: a serpentine path that visits every site of an X×Y×Z box once.

    Rows run back and forth in x, rows stack in y, layers stack in z (each layer reversed so the path stays
    connected). Returns the residue coordinates r_{ia}, shape (L, 3) with L = X·Y·Z.
    """
    X, Y, Z = dims
    coords = []
    for z in range(Z):
        ys = range(Y) if z % 2 == 0 else range(Y - 1, -1, -1)
        for row, y in enumerate(ys):
            xs = range(X) if (row + z * Y) % 2 == 0 else range(X - 1, -1, -1)
            coords += [(x, y, z) for x in xs]
    return torch.tensor(coords, device=device)


def contact_map(coords: torch.Tensor, min_sep: int = 3) -> torch.Tensor:
    """True contacts (L, L) bool: lattice neighbours (city-block distance 1) at least min_sep apart in sequence."""
    L = coords.shape[0]
    dist = (coords[:, None, :] - coords[None, :, :]).abs().sum(-1)  # (L, L) city-block distance
    idx = torch.arange(L, device=coords.device)
    far_in_sequence = (idx[:, None] - idx[None, :]).abs() >= min_sep
    return (dist == 1) & far_in_sequence


def parity(coords: torch.Tensor) -> torch.Tensor:
    """Checkerboard colour (L,) bool: x+y+z even. Lattice neighbours always have opposite colours."""
    return coords.sum(-1) % 2 == 0


def planted_potts(contacts: torch.Tensor, q: int, coupling: float, field: float, seed: int):
    """Random Potts parameters with couplings ONLY on the true contacts.

    h_{ia}:     (L, q)        site fields ~ field · N(0, 1)            (some residues more conserved than others)
    J_{iajb}:   (L, q, L, q)  for each contact a random q×q block ~ coupling · N(0, 1), J_{iajb} = J_{jbia}
    """
    gen = torch.Generator().manual_seed(seed)
    L, dev = contacts.shape[0], contacts.device
    h = field * torch.randn(L, q, generator=gen)
    J = torch.zeros(L, q, L, q)
    for i, j in torch.nonzero(torch.triu(contacts.cpu(), 1)).tolist():
        block = coupling * torch.randn(q, q, generator=gen)
        J[i, :, j, :] = block
        J[j, :, i, :] = block.T  # symmetric: the energy of pair (i, j) is the same seen from either side
    return h.to(dev), J.to(dev)


def one_hot(seqs: torch.Tensor, q: int) -> torch.Tensor:
    """Sequences (N, L) of integer states 0…q-1 → one-hot x_{nia}, shape (N, L, q), float."""
    return torch.nn.functional.one_hot(seqs, q).float()


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/bioinformatics/coevolution_contacts/solve.py
"""Sample the alignment from the planted Potts model, then infer contacts with mean-field DCA or pseudo-likelihood."""

import torch

from problems._common.progress import steps

from .equation import (
    apc,
    covariance,
    frobenius_scores,
    local_field,
    mean_field_couplings,
    pair_frequencies,
    pseudo_log_likelihood,
    single_frequencies,
    with_pseudocount,
)
from .tensors import one_hot


def gibbs_sample(h, J, colour, n_seq: int, n_sweeps: int, seed: int) -> torch.Tensor:
    """N independent Markov chains (one per sequence), updated together. Returns the final sequences (N, L).

    Checkerboard Gibbs sampling: contacts only join sites of opposite colour, so all sites of one colour are
    conditionally independent given the others and can be resampled AT ONCE from softmax_a E_{nia}.
    The Gumbel-max trick draws from a softmax: argmax_a (E_{nia} + g_{nia}), g = −log(−log U).
    """
    gen = torch.Generator(device=h.device).manual_seed(seed)
    L, q = h.shape
    s = torch.randint(q, (n_seq, L), generator=gen, device=h.device)  # random start
    for _ in steps(n_sweeps, "Gibbs sweeps"):
        for mask in (colour, ~colour):
            E = local_field(one_hot(s, q), h, J)  # (N, L, q)
            u = torch.rand(E.shape, generator=gen, device=h.device).clamp_min(1e-12)
            new = torch.argmax(E - torch.log(-torch.log(u)), dim=-1)  # (N, L) sample from softmax_a E
            s = torch.where(mask, new, s)  # only this colour changes
    return s


def covariance_scores(x: torch.Tensor, lam: float) -> torch.Tensor:
    """Baseline: APC-corrected Frobenius norm of the raw covariance C_ij (direct + indirect correlations)."""
    f1, f2 = with_pseudocount(single_frequencies(x), pair_frequencies(x), lam)
    return apc(frobenius_scores(covariance(f1, f2)))


def mfdca_scores(x: torch.Tensor, lam: float) -> torch.Tensor:
    """Mean-field DCA: frequencies → covariance → J = −C⁻¹ → Frobenius norm → APC.   (L, L)."""
    f1, f2 = with_pseudocount(single_frequencies(x), pair_frequencies(x), lam)
    return apc(frobenius_scores(mean_field_couplings(covariance(f1, f2))))


def train_plm(x: torch.Tensor, n_iter: int, lr: float, reg: float, n_snapshots: int = 40):
    """Pseudo-likelihood maximisation with autograd and Adam.

    loss = −(1/N) Σ_{n,i} log P(s_ni | s_n,rest) + reg · Σ J²   (the L2 penalty stands in for a prior).
    J is stored unconstrained and symmetrised, with the diagonal blocks masked out, before each use.
    Returns the final scores (L, L), the snapshot iterations and score snapshots (S, L, L), and the loss history.
    """
    N, L, q = x.shape
    h = torch.zeros(L, q, device=x.device, requires_grad=True)
    W = torch.zeros(L, q, L, q, device=x.device, requires_grad=True)
    off_diag = (1 - torch.eye(L, device=x.device))[:, None, :, None]  # (L, 1, L, 1)
    opt = torch.optim.Adam([h, W], lr=lr)

    def couplings() -> torch.Tensor:
        return 0.5 * (W + W.permute(2, 3, 0, 1)) * off_diag  # J_iajb = J_jbia, J_iaib = 0

    snap_iters = sorted({round(s * (n_iter - 1) / max(1, n_snapshots - 1)) for s in range(n_snapshots)})
    snaps, losses = [], torch.zeros(n_iter)
    for it in steps(n_iter, "pseudo-likelihood"):
        J = couplings()
        loss = -pseudo_log_likelihood(x, h, J) + reg * (J**2).sum()
        opt.zero_grad()
        loss.backward()
        opt.step()
        losses[it] = loss.item()
        if it in snap_iters:
            with torch.no_grad():
                snaps.append(apc(frobenius_scores(couplings())).cpu())
    return snaps[-1], snap_iters, torch.stack(snaps), losses


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/bioinformatics/coevolution_contacts/plot.py
"""Animate contact prediction: the alignment, the top-L predicted contacts on the true map, and precision."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.colors import ListedColormap

from problems._common.plotting import frame_indices, save_gif

from .equation import top_pairs

BLUE, RED, ORANGE, GREY = "#2563eb", "#dc2626", "#ea580c", "#9ca3af"


def animate(
    seqs: torch.Tensor,
    q: int,
    contacts: torch.Tensor,
    main_scores: torch.Tensor,
    cov_scores: torch.Tensor,
    xs: list[int],
    prec_main: list[float],
    prec_cov: list[float],
    random_level: float,
    label: str,
    depth_sweep: bool,
    min_sep: int,
    path: Path,
) -> Path:
    L = contacts.shape[0]
    n_rows = min(48, seqs.shape[0])
    letters = ListedColormap(plt.get_cmap("tab20")(np.arange(q) % 20)).with_extremes(bad="white")  # bad = unused rows
    msa = seqs[:n_rows].numpy().astype(float)  # (rows, L)

    fig = plt.figure(figsize=(9.6, 3.9))
    grid = fig.add_gridspec(1, 3, width_ratios=[1.0, 1.25, 1.25])
    ax_msa, ax_map, ax_p = (fig.add_subplot(grid[0, k]) for k in range(3))

    # --- the alignment: one row per sequence, one colour per letter ---
    msa_img = ax_msa.imshow(msa, cmap=letters, vmin=-0.5, vmax=q - 0.5, aspect="auto",
                            interpolation="nearest")  # fmt: skip
    ax_msa.set_xlabel("column i")
    ax_msa.set_ylabel("sequence n")
    ax_msa.grid(False)
    ax_msa.set_title(f"MSA (first {n_rows} rows)", fontsize=10)
    col_marks = [ax_msa.axvline(-5, color="#f59e0b", lw=2) for _ in range(2)]

    # --- contact map: true contacts (grey), predictions upper = DCA/PLM, lower = covariance ---
    ci, cj = np.nonzero(contacts.numpy())
    ax_map.scatter(cj, ci, s=22, marker="s", color="#d1d5db", lw=0, label="true contact")
    ax_map.plot([0, L - 1], [0, L - 1], color=GREY, lw=0.8)
    ax_map.set_xlim(-0.5, L - 0.5)
    ax_map.set_ylim(L - 0.5, -0.5)
    ax_map.set_aspect("equal")
    ax_map.set_title(f"top-L: {label} (upper)\ncovariance (lower)", fontsize=9)
    ax_map.tick_params(labelsize=8)
    hit_up = ax_map.scatter([], [], s=10, color=BLUE, label="correct")
    miss_up = ax_map.scatter([], [], s=10, color=RED, label="wrong")
    hit_lo = ax_map.scatter([], [], s=10, color=BLUE)
    miss_lo = ax_map.scatter([], [], s=10, color=RED)
    ax_map.legend(loc="lower left", fontsize=6.5, framealpha=0.9)

    # --- precision of the top-L pairs ---
    ax_p.set_ylim(0, 1.05)
    ax_p.set_ylabel("top-L precision")
    if depth_sweep:
        ax_p.set_xscale("log")
        ticks = [t for t in (20, 50, 100, 200, 500, 1000, 2000, 4000) if xs[0] <= t <= xs[-1]]
        ax_p.set_xticks(ticks, [str(t) for t in ticks])
        ax_p.minorticks_off()
        ax_p.set_xlabel("sequences in the MSA, N")
    else:
        ax_p.set_xlabel("Adam iteration")
    ax_p.set_xlim(xs[0], xs[-1])
    ax_p.axhline(random_level, color=GREY, ls="--", lw=1)
    ax_p.text(xs[-1], random_level + 0.02, "random", ha="right", fontsize=8, color="#6b7280")
    (line_main,) = ax_p.plot([], [], "o-", color=BLUE, ms=3, lw=1.6, label=label)
    (line_cov,) = ax_p.plot([], [], "s-" if depth_sweep else "-", color=ORANGE, ms=3, lw=1.4, label="covariance")
    ax_p.legend(loc="center right", fontsize=8)
    ax_p.set_title("precision vs true contacts", fontsize=10)
    counter = ax_p.text(0.03, 0.1, "", transform=ax_p.transAxes, fontsize=9)

    def split(pairs: torch.Tensor, upper: bool):
        i, j = pairs[:, 0].numpy(), pairs[:, 1].numpy()
        ok = contacts[pairs[:, 0], pairs[:, 1]].numpy()
        xy = np.stack([j, i], 1) if upper else np.stack([i, j], 1)  # (column, row) for the scatter
        return xy[ok], xy[~ok]

    def update(s: int):
        top = top_pairs(main_scores[s], L, min_sep)
        good, bad = split(top, True)
        hit_up.set_offsets(good.reshape(-1, 2))
        miss_up.set_offsets(bad.reshape(-1, 2))
        good, bad = split(top_pairs(cov_scores[s], L, min_sep), False)
        hit_lo.set_offsets(good.reshape(-1, 2))
        miss_lo.set_offsets(bad.reshape(-1, 2))
        for mark, col in zip(col_marks, top[0].tolist()):  # the strongest predicted pair, in the MSA
            mark.set_xdata([col, col])
        if depth_sweep:
            shown = msa.copy()
            shown[xs[s] :] = np.nan  # only the first N sequences are used in this frame
            msa_img.set_data(shown)
            counter.set_text(f"N = {xs[s]}")
        else:
            counter.set_text(f"iteration {xs[s] + 1}")
        line_main.set_data(xs[: s + 1], prec_main[: s + 1])
        line_cov.set_data(xs[: s + 1], prec_cov[: s + 1])

    fig.tight_layout()
    frames = frame_indices(len(xs), 60)
    frames = frames + [frames[-1]] * (len(frames) // 3)  # hold the final picture
    return save_gif(fig, update, frames, path, fps=8)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'fold': '4x4x4',
    'q': 5,
    'coupling': 1.5,
    'n_seq': 1000,
    'sweeps': 200,
    'pseudocount': 0.3,
    'method': 'mfdca',
    'plm_iters': 300
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.bioinformatics.coevolution_contacts")]:
    del sys.modules[name]
main = importlib.import_module("problems.bioinformatics.coevolution_contacts.main")
problem_dir = Path("problems/bioinformatics/coevolution_contacts")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
